# LAB – Clustering (IA não supervisionada): DBSCAN com o dataset Wholesale Customers

**PUC-SP – Ciência da Computação – Inteligência Artificial (IA ML)**

**Alunos (dupla):**
- Luan Bonasorte Capella
- Matheus Heimrath Barbosa

**Objetivo:** experimentar e explorar um modelo de *clustering* (agrupamento) baseado em densidade com Scikit-Learn: o **DBSCAN** (`DBSCAN`).

**Dataset:** [Wholesale Customers Data Set](https://www.kaggle.com/datasets/binovi/wholesale-customers-data-set) do Kaggle
(originalmente do UCI Machine Learning Repository). São os gastos anuais de 440 clientes de um distribuidor atacadista em Portugal,
separados por categoria de produto (frescos, laticínios, mercearia, congelados, limpeza/papel e delicatessen).

**Problema:** aprendizado **não supervisionado**. Queremos descobrir **segmentos de clientes** com padrões de compra parecidos e,
ao mesmo tempo, identificar os **clientes atípicos**, aqueles que não se encaixam em nenhum padrão. O DBSCAN faz as duas coisas:
ele encontra os grupos sozinho (sem precisar informar o número de clusters) e marca como **ruído** os pontos isolados.

## 1. Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score

RANDOM_STATE = 42
CORES = ["#4C72B0", "#55A868", "#C44E52", "#8172B2", "#DD8452", "#937860", "#DA8BC3", "#CCB974", "#64B5CD"]
COR_RUIDO = "#8C8C8C"

## 2. Carregando os dados

Baixe o arquivo `Wholesale customers data.csv` na página do dataset no Kaggle e coloque na mesma pasta do notebook.
Se o arquivo não for encontrado, o código tenta baixar automaticamente com o pacote `kagglehub` (precisa de `pip install kagglehub`).

In [ ]:
CAMINHO = "Wholesale customers data.csv"

try:
    df = pd.read_csv(CAMINHO)
except FileNotFoundError:
    import kagglehub
    pasta = kagglehub.dataset_download("binovi/wholesale-customers-data-set")
    df = pd.read_csv(f"{pasta}/Wholesale customers data.csv")

print("Dimensões:", df.shape)
df.head()

**Dicionário das colunas** (valores de gasto em unidades monetárias por ano):

| Coluna | Significado |
|---|---|
| `Channel` | Canal do cliente: 1 = Horeca (hotéis, restaurantes e cafés), 2 = Varejo |
| `Region` | Região: 1 = Lisboa, 2 = Porto, 3 = Outras |
| `Fresh` | Gasto com produtos frescos |
| `Milk` | Gasto com laticínios |
| `Grocery` | Gasto com mercearia |
| `Frozen` | Gasto com congelados |
| `Detergents_Paper` | Gasto com produtos de limpeza e papel |
| `Delicassen` | Gasto com delicatessen |

`Channel` e `Region` **não** entram no modelo. Usamos apenas os 6 tipos de gasto, e o `Channel` fica guardado para conferir,
no final, se os grupos encontrados têm relação com o tipo de cliente.

## 3. Análise exploratória

In [ ]:
df.info()

In [ ]:
print("Valores ausentes por coluna:")
print(df.isna().sum())
print("\nLinhas duplicadas:", df.duplicated().sum())

In [ ]:
df.describe().round(1)

Observações:
- O dataset não tem valores ausentes.
- A **média** de cada categoria é bem maior que a **mediana** (o valor de 50%), e o máximo é muito distante do restante.
  Isso indica que alguns poucos clientes compram muito mais que os outros.

In [ ]:
gastos = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]

fig, axes = plt.subplots(2, 3, figsize=(13, 6.5))
for ax, col in zip(axes.ravel(), gastos):
    ax.hist(df[col], bins=30, color="#4C72B0", edgecolor="white")
    ax.set_title(col)
plt.suptitle("Distribuição dos gastos anuais por categoria")
plt.tight_layout()
plt.show()

print("Assimetria (skewness) de cada categoria:")
print(df[gastos].skew().round(2).sort_values(ascending=False))

Todas as categorias têm **cauda longa à direita**: a maioria dos clientes gasta pouco e poucos clientes gastam valores enormes.
Isso é um problema para o DBSCAN, que depende de distâncias: os clientes grandes ficariam tão longe dos demais que a "densidade"
dos dados seria distorcida.

In [ ]:
corr = df[gastos].corr()

fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(gastos)), gastos, rotation=45, ha="right")
ax.set_yticks(range(len(gastos)), gastos)
for i in range(len(gastos)):
    for j in range(len(gastos)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im)
ax.set_title("Correlação entre os gastos")
plt.tight_layout()
plt.show()

`Grocery`, `Milk` e `Detergents_Paper` aparecem bem correlacionados: clientes que compram muito de mercearia costumam comprar
também laticínios e produtos de limpeza. Esse é um padrão típico de **varejo** (mercados que revendem esses produtos).

In [ ]:
nomes_canal = {1: "Horeca", 2: "Varejo"}
print("Clientes por canal:")
print(df["Channel"].map(nomes_canal).value_counts())

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ["Fresh", "Grocery", "Detergents_Paper"]):
    dados = [df.loc[df["Channel"] == c, col] for c in [1, 2]]
    caixa = ax.boxplot(dados, patch_artist=True)
    for patch, cor in zip(caixa["boxes"], ["#DD8452", "#4C72B0"]):
        patch.set_facecolor(cor)
    ax.set_xticks([1, 2], ["Horeca", "Varejo"])
    ax.set_yscale("log")
    ax.set_title(f"{col} (escala log)")
plt.suptitle("Os canais têm padrões de compra diferentes")
plt.tight_layout()
plt.show()

Hotéis, restaurantes e cafés (Horeca) tendem a gastar mais com **produtos frescos**, enquanto o varejo gasta mais com
**mercearia** e **limpeza/papel**. O modelo não vai ver essa coluna, mas ela servirá para conferir se os grupos fazem sentido.

## 4. Pré-processamento

- **Transformação logarítmica (`log1p`)** em todas as categorias de gasto, para reduzir a cauda longa. Assim, a diferença entre
  gastar 1.000 e 10.000 passa a ter o mesmo peso que a diferença entre 10.000 e 100.000.
- **Padronização (`StandardScaler`)**: o DBSCAN mede distância euclidiana, então todas as variáveis precisam estar na mesma escala.

In [ ]:
X = np.log1p(df[gastos])

escalador = StandardScaler()
X_scaled = escalador.fit_transform(X)

print("Assimetria depois do log:")
print(X.skew().round(2))
print("\nMédia depois da padronização:", X_scaled.mean(axis=0).round(2))
print("Desvio padrão depois da padronização:", X_scaled.std(axis=0).round(2))

### PCA para visualização

Usamos **PCA** só para projetar as 6 dimensões em 2 e conseguir **ver** os grupos. O DBSCAN é treinado com as 6 variáveis padronizadas.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

print("Variância explicada por componente:", pca.explained_variance_ratio_.round(3))
print(f"Total explicado pelas 2 componentes: {pca.explained_variance_ratio_.sum():.1%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True, sharey=True)
axes[0].scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.6, c=COR_RUIDO)
axes[0].set_title("Clientes projetados em 2D (sem rótulos)")
for c, cor in [(1, "#DD8452"), (2, "#4C72B0")]:
    m = df["Channel"] == c
    axes[1].scatter(X_pca[m, 0], X_pca[m, 1], alpha=0.6, c=cor, label=nomes_canal[c])
axes[1].set_title("Mesmo gráfico colorido pelo canal (só para referência)")
axes[1].legend()
for ax in axes:
    ax.set_xlabel("PC1")
axes[0].set_ylabel("PC2")
plt.tight_layout()
plt.show()

### Função auxiliar

Uma função para desenhar o resultado do DBSCAN no plano do PCA. Os pontos de **ruído** (rótulo `-1`) aparecem em cinza com um "x".

In [ ]:
def plot_dbscan(labels, titulo):
    fig, ax = plt.subplots(figsize=(8, 5.5))
    grupos = [g for g in np.unique(labels) if g != -1]
    for i, g in enumerate(grupos):
        m = labels == g
        ax.scatter(X_pca[m, 0], X_pca[m, 1], alpha=0.75, c=CORES[i % len(CORES)], label=f"Grupo {g} ({m.sum()})")
    m = labels == -1
    ax.scatter(X_pca[m, 0], X_pca[m, 1], c=COR_RUIDO, marker="x", label=f"Ruído ({m.sum()})")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
    ax.set_title(titulo)
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

## 5. Modelo – DBSCAN (`DBSCAN`)

**DBSCAN** significa *Density-Based Spatial Clustering of Applications with Noise*. Em vez de procurar centros (como o K-Means),
ele procura **regiões densas** de pontos.

**Como funciona:**
1. Para cada ponto, o algoritmo conta quantos vizinhos existem dentro de um raio **`eps`**.
2. Se houver pelo menos **`min_samples`** pontos nesse raio (contando o próprio ponto), ele é um **ponto central** (*core point*).
3. Pontos centrais que estão ao alcance uns dos outros são ligados e formam um **cluster**.
4. Pontos que não são centrais, mas estão no raio de algum ponto central, entram no cluster como **pontos de borda**.
5. Pontos que não são alcançados por nenhum ponto central viram **ruído** e recebem o rótulo `-1`.

**Vantagens:** não precisa informar o número de grupos, encontra grupos de **formato qualquer** e separa os outliers automaticamente.
**Desvantagem:** o resultado depende muito da escolha de `eps` e `min_samples`, e ele tem dificuldade quando os grupos têm densidades muito diferentes.

### 5.1 Escolhendo `min_samples`

Uma regra prática comum é usar **`min_samples = 2 × número de dimensões`**. Com 6 variáveis, isso dá `min_samples = 12`.
Valores maiores deixam o modelo mais exigente (mais pontos viram ruído) e valores menores criam grupos pequenos demais.

In [ ]:
MIN_SAMPLES = 2 * X_scaled.shape[1]
print("min_samples =", MIN_SAMPLES)

### 5.2 Escolhendo `eps`: gráfico de distância k

Para cada cliente calculamos a distância até o seu `min_samples`-ésimo vizinho mais próximo e ordenamos essas distâncias.
- Pontos dentro de regiões densas têm distância **pequena**.
- Outliers têm distância **grande**.

A curva fica quase plana e depois sobe rapidamente. O **"joelho"** da curva, onde ela começa a subir, é um bom candidato para `eps`.
Para achar o joelho de forma objetiva, usamos o ponto da curva mais distante da reta que liga o primeiro ao último ponto.

In [ ]:
vizinhos = NearestNeighbors(n_neighbors=MIN_SAMPLES).fit(X_scaled)
distancias, _ = vizinhos.kneighbors(X_scaled)
k_dist = np.sort(distancias[:, -1])

# Joelho: ponto mais distante da reta entre o primeiro e o último ponto (com os eixos normalizados em 0-1)
x_norm = np.linspace(0, 1, len(k_dist))
y_norm = (k_dist - k_dist.min()) / (k_dist.max() - k_dist.min())
distancia_reta = np.abs(y_norm - x_norm) / np.sqrt(2)
idx_joelho = distancia_reta.argmax()
EPS_JOELHO = k_dist[idx_joelho]

plt.figure(figsize=(8, 4.5))
plt.plot(k_dist, color="#4C72B0")
plt.axhline(EPS_JOELHO, color="#C44E52", linestyle="--", label=f"joelho: eps ≈ {EPS_JOELHO:.2f}")
plt.scatter(idx_joelho, EPS_JOELHO, color="#C44E52", zorder=3)
plt.xlabel("Clientes ordenados pela distância")
plt.ylabel(f"Distância até o {MIN_SAMPLES}º vizinho")
plt.title("Gráfico de distância k")
plt.legend()
plt.tight_layout()
plt.show()

print(f"eps sugerido pelo joelho: {EPS_JOELHO:.3f}")

### 5.3 Experimentando combinações de `eps` e `min_samples`

O joelho e a regra do `2 × dimensões` são só pontos de partida. Testamos uma grade de valores próximos e, para cada combinação, observamos:
- o **número de grupos** encontrados;
- a **porcentagem de ruído**;
- o **silhouette** calculado só com os pontos que receberam grupo (o ruído não pertence a nenhum grupo, por isso fica de fora).

In [ ]:
valores_eps = np.round(np.linspace(0.4 * EPS_JOELHO, 1.6 * EPS_JOELHO, 25), 3)
valores_min = [4, 6, 8, 10, MIN_SAMPLES, 15, 20]

resultados = []
for ms in valores_min:
    for eps in valores_eps:
        r = DBSCAN(eps=eps, min_samples=ms).fit_predict(X_scaled)
        n_grupos = len(set(r) - {-1})
        agrupados = r != -1
        sil = silhouette_score(X_scaled[agrupados], r[agrupados]) if n_grupos >= 2 else np.nan
        resultados.append({"min_samples": ms, "eps": eps, "Nº de grupos": n_grupos,
                           "% ruído": 100 * (r == -1).mean(), "Silhouette": sil})

tabela_params = pd.DataFrame(resultados)
print(f"{len(tabela_params)} combinações testadas")
print("Combinações com 2 ou mais grupos:", (tabela_params["Nº de grupos"] >= 2).sum())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
dados_ms = tabela_params[tabela_params["min_samples"] == MIN_SAMPLES]
ax.plot(dados_ms["eps"], dados_ms["% ruído"], "o-", color=COR_RUIDO)
ax.set_xlabel("eps"); ax.set_ylabel("% de pontos como ruído")
ax2 = ax.twinx()
ax2.step(dados_ms["eps"], dados_ms["Nº de grupos"], where="mid", color="#4C72B0")
ax2.set_ylabel("Nº de grupos", color="#4C72B0")
ax.axvline(EPS_JOELHO, color="#C44E52", linestyle="--")
plt.title(f"Efeito do eps (min_samples = {MIN_SAMPLES}): raio pequeno = muito ruído; raio grande = um grupo só")
plt.tight_layout()
plt.show()

**Como ler:** com `eps` pequeno, quase nenhum ponto tem vizinhos suficientes e muitos clientes viram ruído. Conforme `eps` cresce,
o ruído diminui e os grupos vão se unindo, até que, com `eps` grande, sobra um único grupo.

**Regra de escolha:** entre as combinações com **pelo menos 2 grupos** e **no máximo 20% de ruído**, ficamos com a de **maior silhouette**.
Isso evita escolher parâmetros que separem bem os grupos, mas joguem boa parte dos clientes no ruído.
Se nenhuma combinação atender a essas condições, usamos o `eps` do joelho com `min_samples = 12`.

Nesse último caso, o resultado será **um grupo principal + ruído**. Isso também é uma informação: significa que, nas 6 dimensões,
os clientes formam uma nuvem contínua (sem "vales" de baixa densidade entre segmentos), e o DBSCAN passa a funcionar como um
**detector de clientes atípicos**.

In [ ]:
candidatos = tabela_params[(tabela_params["Nº de grupos"] >= 2) & (tabela_params["% ruído"] <= 20)]

if candidatos["Silhouette"].notna().any():
    melhor = candidatos.loc[candidatos["Silhouette"].idxmax()]
    EPS, MIN_SAMPLES_FINAL = float(melhor["eps"]), int(melhor["min_samples"])
else:
    EPS, MIN_SAMPLES_FINAL = round(float(EPS_JOELHO), 3), MIN_SAMPLES

print(f"eps escolhido: {EPS}")
print(f"min_samples escolhido: {MIN_SAMPLES_FINAL}")
print("\nMelhores combinações candidatas:")
candidatos.sort_values("Silhouette", ascending=False).head(10).round(3)

### 5.4 Treinando o modelo final

In [ ]:
dbscan = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES_FINAL)
rotulos = dbscan.fit_predict(X_scaled)
df["cluster"] = rotulos

n_grupos = len(set(rotulos) - {-1})
n_ruido = (rotulos == -1).sum()
n_centrais = len(dbscan.core_sample_indices_)

print(f"Grupos encontrados: {n_grupos}")
print(f"Pontos centrais: {n_centrais}")
print(f"Pontos de borda: {len(rotulos) - n_centrais - n_ruido}")
print(f"Ruído: {n_ruido} clientes ({n_ruido / len(rotulos):.1%})")
print("\nClientes por grupo (-1 = ruído):")
print(df["cluster"].value_counts().sort_index())

agrupados = rotulos != -1
if n_grupos >= 2:
    print(f"\nSilhouette (sem o ruído): {silhouette_score(X_scaled[agrupados], rotulos[agrupados]):.3f}")
    print(f"Davies-Bouldin (sem o ruído): {davies_bouldin_score(X_scaled[agrupados], rotulos[agrupados]):.3f}")

In [ ]:
plot_dbscan(rotulos, f"DBSCAN (eps = {EPS}, min_samples = {MIN_SAMPLES_FINAL}) no plano do PCA")

### 5.5 Interpretando os grupos

Para entender o que caracteriza cada grupo, olhamos a **mediana** dos gastos nos valores originais.
Usamos a mediana, e não a média, porque os gastos têm valores extremos que distorceriam a média.

In [ ]:
perfil = df.groupby("cluster")[gastos].median().round(0)
perfil["Nº de clientes"] = df["cluster"].value_counts().sort_index()
perfil.index = perfil.index.map(lambda g: "Ruído" if g == -1 else f"Grupo {g}")
perfil

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
grupos_ordem = sorted(df["cluster"].unique(), key=lambda g: (g == -1, g))
rotulos_eixo = ["Ruído" if g == -1 else f"G{g}" for g in grupos_ordem]
cores_caixas = [COR_RUIDO if g == -1 else CORES[i % len(CORES)] for i, g in enumerate(grupos_ordem)]

for ax, col in zip(axes.ravel(), gastos):
    dados = [df.loc[df["cluster"] == g, col] for g in grupos_ordem]
    caixa = ax.boxplot(dados, patch_artist=True)
    for patch, cor in zip(caixa["boxes"], cores_caixas):
        patch.set_facecolor(cor)
    ax.set_xticks(range(1, len(grupos_ordem) + 1), rotulos_eixo)
    ax.set_yscale("log")
    ax.set_title(f"{col} (escala log)")
plt.suptitle("Gastos por grupo encontrado pelo DBSCAN")
plt.tight_layout()
plt.show()

### 5.6 Os clientes atípicos (ruído)

Uma das grandes vantagens do DBSCAN é apontar os clientes que **não seguem nenhum padrão**. Eles podem ser clientes muito grandes,
clientes que compram só uma categoria ou até erros de registro. Para uma empresa, vale a pena olhar esses casos individualmente.

Abaixo mostramos os clientes de ruído e, para cada um, a categoria em que o gasto mais se afasta do normal
(maior distância em desvios padrão, já na escala log).

In [ ]:
ruido = df[df["cluster"] == -1].copy()

if len(ruido) > 0:
    z = pd.DataFrame(X_scaled, columns=gastos, index=df.index).loc[ruido.index]
    ruido["Categoria mais atípica"] = z.abs().idxmax(axis=1)
    ruido["Desvios padrão"] = z.abs().max(axis=1).round(2)
    valor_atipico = [z.loc[i, c] for i, c in zip(z.index, ruido["Categoria mais atípica"])]
    ruido["Direção"] = np.where(np.array(valor_atipico) > 0, "acima do normal", "abaixo do normal")
    ruido["Channel"] = ruido["Channel"].map(nomes_canal)
    print(f"{len(ruido)} clientes atípicos. Os 15 mais extremos:")
    mostrar = ruido.sort_values("Desvios padrão", ascending=False)
    mostrar = mostrar[["Channel"] + gastos + ["Categoria mais atípica", "Direção", "Desvios padrão"]].head(15)
else:
    print("Nenhum cliente foi classificado como ruído com esses parâmetros.")
    mostrar = ruido
mostrar

In [ ]:
if len(ruido) > 0:
    contagem = ruido["Categoria mais atípica"].value_counts().reindex(gastos, fill_value=0)
    contagem.plot(kind="bar", color="#C44E52", figsize=(8, 4))
    plt.title("Em qual categoria os clientes atípicos mais se destacam?")
    plt.ylabel("Nº de clientes de ruído")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

### 5.7 Os grupos têm relação com o canal do cliente?

O `Channel` não foi usado no treino. Se os grupos do DBSCAN coincidirem com Horeca e Varejo, isso indica que o modelo encontrou
um padrão real de compra. O **Adjusted Rand Index (ARI)** mede essa concordância (1 = idêntico, 0 = igual ao acaso),
calculado apenas com os clientes que receberam grupo.

In [ ]:
tabela_canal = pd.crosstab(df["cluster"].map(lambda g: "Ruído" if g == -1 else f"Grupo {g}"),
                           df["Channel"].map(nomes_canal), rownames=["Cluster"], colnames=["Canal"])

if n_grupos >= 2:
    ari = adjusted_rand_score(df.loc[agrupados, "Channel"], rotulos[agrupados])
    print(f"ARI entre os grupos do DBSCAN e o canal (sem o ruído): {ari:.3f}\n")
tabela_canal

In [ ]:
(tabela_canal.div(tabela_canal.sum(axis=1), axis=0) * 100).plot(
    kind="barh", stacked=True, color=["#DD8452", "#4C72B0"], figsize=(8, 3.5))
plt.xlabel("% dos clientes do grupo")
plt.title("Composição de cada grupo por canal")
plt.legend(title="Canal", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### 5.8 Sensibilidade aos parâmetros

Para entender melhor o comportamento do DBSCAN, variamos `eps` e `min_samples` ao mesmo tempo e observamos o número de grupos
e a porcentagem de ruído. O mapa mostra como uma pequena mudança nos parâmetros pode mudar bastante o resultado.

In [ ]:
grade_eps = np.round(np.linspace(0.6 * EPS_JOELHO, 1.6 * EPS_JOELHO, 8), 2)
grade_min = valores_min

mapa_grupos = np.zeros((len(grade_min), len(grade_eps)), dtype=int)
mapa_ruido = np.zeros((len(grade_min), len(grade_eps)))
for i, ms in enumerate(grade_min):
    for j, eps in enumerate(grade_eps):
        r = DBSCAN(eps=eps, min_samples=ms).fit_predict(X_scaled)
        mapa_grupos[i, j] = len(set(r) - {-1})
        mapa_ruido[i, j] = 100 * (r == -1).mean()

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
for ax, mapa, titulo, cmap, fmt in [(axes[0], mapa_grupos, "Nº de grupos", "Blues", "{:d}"),
                                    (axes[1], mapa_ruido, "% de ruído", "Reds", "{:.0f}")]:
    im = ax.imshow(mapa, cmap=cmap, aspect="auto")
    ax.set_xticks(range(len(grade_eps)), grade_eps)
    ax.set_yticks(range(len(grade_min)), grade_min)
    ax.set_xlabel("eps"); ax.set_ylabel("min_samples")
    ax.set_title(titulo)
    for a in range(mapa.shape[0]):
        for b in range(mapa.shape[1]):
            ax.text(b, a, fmt.format(mapa[a, b]), ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=ax)
plt.suptitle("Sensibilidade do DBSCAN a eps e min_samples")
plt.tight_layout()
plt.show()

## Referências

- Scikit-Learn – Clustering: https://scikit-learn.org/stable/modules/clustering.html
- Scikit-Learn – DBSCAN (guia): https://scikit-learn.org/stable/modules/clustering.html#dbscan
- DBSCAN (API): https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html
- NearestNeighbors: https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.NearestNeighbors.html
- Ester, M. et al. (1996). *A Density-Based Algorithm for Discovering Clusters in Large Spatial Databases with Noise*. KDD-96.
- Kaggle – Wholesale Customers Data Set: https://www.kaggle.com/datasets/binovi/wholesale-customers-data-set
- UCI – Wholesale customers: https://archive.ics.uci.edu/dataset/292/wholesale+customers